# bursakit quickstart

Bursa Malaysia stock data and Malaysian macro series as pandas DataFrames.

```bash
pip install bursakit matplotlib
```

Every call fetches from the source the first time and reads from a local cache after that. Stock data comes from Yahoo Finance and is for your own personal, non-commercial use; see [the source terms](https://github.com/MaosterM/bursakit/blob/main/docs/source-terms.md).

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

import bursakit as bk

## Find a stock

A code, a `.KL` symbol or a name all resolve to the same ticker.

In [ ]:
[bk.resolve(x) for x in ("1155", "1155.KL", "maybank", "tenaga")]

## Daily prices

In [ ]:
maybank = bk.prices("MAYBANK", start="2021-01-01")
maybank.tail()

Compare three banks, each rebased to 1 at the start.

In [ ]:
banks = ["MAYBANK", "PBBANK", "CIMB"]
closes = pd.DataFrame({name: bk.prices(name, start="2021-01-01")["close"] for name in banks})
(closes / closes.iloc[0]).plot(title="Bank shares since 2021, rebased to 1", figsize=(9, 4));

## Dividends

In [ ]:
dividends = bk.dividends("MAYBANK")
dividends.tail()

In [ ]:
# Dividends per share by year, in MYR
dividends["amount"].groupby(dividends.index.year).sum().tail(10)

## Everything listed

In [ ]:
listed = bk.listings()
print(len(listed), "stocks")
listed["sector"].value_counts()

## Macro: the OPR and the ringgit

From Bank Negara Malaysia. These can be shared and used commercially with the credit line in `df.attrs`.

In [ ]:
opr = bk.macro.opr(start="2015-01-01")
usd = bk.macro.fx("USD", start="2015-01-01")

# The OPR only has a row per decision; carry the last one up to today.
rate = opr["rate"].reindex(opr.index.union([usd.index[-1]])).ffill()

fig, (top, bottom) = plt.subplots(2, 1, figsize=(9, 5), sharex=True)
top.step(rate.index, rate, where="post", color="#1f6f8b")
top.set_ylabel("OPR, %")
top.set_title("Overnight Policy Rate and USD/MYR since 2015", loc="left")
bottom.plot(usd.index, usd["mid"], color="#c2410c", linewidth=1)
bottom.set_ylabel("MYR per USD")
for ax in (top, bottom):
    ax.spines[["top", "right"]].set_visible(False)
    ax.grid(alpha=0.3)
fig.tight_layout()

## Where the data came from

In [ ]:
maybank.attrs

In [ ]:
usd.attrs

## The cache

Running this notebook a second time makes almost no requests. `refresh=True` on any call fetches again; `bk.cache.clear()` empties it.

In [ ]:
bk.cache.info()